# Classificação — k-NN (Onda de Calor) e Árvore de Decisão (Risco de Incêndio)

**Sub-tarefa 1:** k-Nearest Neighbors (k-NN) para identificar dias de **risco de onda de calor**.  
**Sub-tarefa 2:** Decision Tree Classifier para avaliar **risco de incêndio** e extrair regras interpretáveis.  
Avaliação via **matriz de confusão**, precisão, recall, F1-Score.

In [1]:
pip install pandas==2.2.2 numpy==1.26.4 scikit-learn==1.5.0 matplotlib==3.9.0 seaborn==0.13.2 plotly==5.22.0 streamlit==1.35.0 tensorflow==2.16.1 torch==2.3.1 torchvision==0.18.1 jupyter==1.0.0 ipykernel==6.29.4 nbformat==5.10.4 scipy==1.13.1 joblib==1.4.2


Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-2.2.2.tar.gz (4.4 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  Preparing metadata (pyproject.toml) did not run successfully.
  exit code: 1
  
  [12 lines of output]
  + meson setup C:\Users\bahgo\AppData\Local\Temp\pip-install-tdacejx1\pandas_564cbc33083b4372b25c685ea3389ce4 C:\Users\bahgo\AppData\Local\Temp\pip-install-tdacejx1\pandas_564cbc33083b4372b25c685ea3389ce4\.mesonpy-s84xz5dc\build -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --vsenv --native-file=C:\Users\bahgo\AppData\Local\Temp\pip-install-tdacejx1\pandas_564cbc33083b4372b25c685ea3389ce4\.mesonpy-s84xz5dc\build\meson-python-native-file.ini
  The Meson build system
  Version: 1.2.1
  Source dir: C:\Users\bahgo\AppData\Local\Temp\pip-install-tdacejx1\pandas_564cbc33083b4372b25c685ea3389ce4
  Build dir: C:\Users\bahgo\AppData\Local\Temp\pip-install-tdacejx1\pandas_564cbc33083b4372b25c685ea3389ce4\.mesonpy-s84xz5dc\build
  Build type: native build
  Project name: pandas
  Project version: 2.2.2
  
  ..\..\meson.build:2:0: ERROR: Could 

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score, roc_curve)
from sklearn.preprocessing import StandardScaler
import joblib

from src.data.preprocessor import get_processed_data

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)
print('OK')

In [ ]:
df = get_processed_data('../data/raw/cerrado_clima_raw.csv')
print(f'Dataset: {df.shape}')
print(f"Risco onda de calor: {df['risco_onda_calor'].value_counts().to_dict()}")
print(f"Risco incêndio:      {df['risco_incendio'].value_counts().to_dict()}")

## PARTE 1 — k-NN: Identificação de Dias de Risco de Onda de Calor

In [ ]:
FEAT_KNN = ['temp_max', 'temp_min', 'umidade_relativa', 'velocidade_vento',
            'radiacao_solar', 'mes_sin', 'mes_cos', 'temp_media_30d']
TARGET_KNN = 'risco_onda_calor'

df_knn = df[FEAT_KNN + [TARGET_KNN]].dropna()
X_knn = df_knn[FEAT_KNN]
y_knn = df_knn[TARGET_KNN]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_knn, y_knn, test_size=0.2, stratify=y_knn, random_state=42
)

scaler_knn = StandardScaler()
X_tr_s = scaler_knn.fit_transform(X_tr)
X_te_s = scaler_knn.transform(X_te)

print(f'Treino: {X_tr.shape} | Positivos: {y_tr.sum()} ({100*y_tr.mean():.1f}%)')
print(f'Teste:  {X_te.shape} | Positivos: {y_te.sum()} ({100*y_te.mean():.1f}%)')

In [ ]:
# Seleciona melhor k via validação cruzada
k_range = range(3, 21, 2)
cv_scores = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean', weights='distance')
    scores = []
    for tr_idx, val_idx in skf.split(X_tr_s, y_tr):
        knn.fit(X_tr_s[tr_idx], y_tr.iloc[tr_idx])
        scores.append(roc_auc_score(y_tr.iloc[val_idx], knn.predict_proba(X_tr_s[val_idx])[:, 1]))
    cv_scores.append(np.mean(scores))

best_k = list(k_range)[np.argmax(cv_scores)]
print(f'Melhor k = {best_k} (AUC CV = {max(cv_scores):.4f})')

plt.figure(figsize=(8, 4))
plt.plot(list(k_range), cv_scores, 'o-', color='steelblue')
plt.axvline(best_k, color='red', linestyle='--', label=f'k={best_k}')
plt.xlabel('k (vizinhos)')
plt.ylabel('AUC (Validação Cruzada)')
plt.title('Seleção do Hiperparâmetro k')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/03_knn_selecao_k.png')
plt.show()

In [ ]:
knn_final = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean', weights='distance')
knn_final.fit(X_tr_s, y_tr)
y_pred_knn = knn_final.predict(X_te_s)
y_prob_knn = knn_final.predict_proba(X_te_s)[:, 1]

print(f'\n=== k-NN (k={best_k}) — Relatório de Classificação ===')
print(classification_report(y_te, y_pred_knn, target_names=['Sem Risco', 'Risco Onda Calor']))
print(f'AUC-ROC: {roc_auc_score(y_te, y_prob_knn):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_te, y_pred_knn)
ConfusionMatrixDisplay(cm, display_labels=['Sem Risco', 'Risco']).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Matriz de Confusão — k-NN (k={best_k})')

fpr, tpr, _ = roc_curve(y_te, y_prob_knn)
auc = roc_auc_score(y_te, y_prob_knn)
axes[1].plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUC = {auc:.4f}')
axes[1].plot([0,1],[0,1],'--',color='gray')
axes[1].set_xlabel('Taxa de Falso Positivo')
axes[1].set_ylabel('Taxa de Verdadeiro Positivo')
axes[1].set_title('Curva ROC — k-NN')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/03_knn_resultados.png')
plt.show()

## PARTE 2 — Árvore de Decisão: Risco de Incêndio

In [ ]:
FEAT_DT = ['temp_max', 'umidade_relativa', 'velocidade_vento',
           'precipitacao', 'precip_7d', 'radiacao_solar', 'estacao_seca']
TARGET_DT = 'risco_incendio'

df_dt = df[FEAT_DT + [TARGET_DT]].dropna()
X_dt = df_dt[FEAT_DT]
y_dt = df_dt[TARGET_DT]

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_dt, y_dt, test_size=0.2, stratify=y_dt, random_state=42
)

print(f'Treino: {X_tr2.shape} | Incêndio positivo: {y_tr2.sum()} ({100*y_tr2.mean():.1f}%)')

In [ ]:
dt = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=50,
    class_weight='balanced',
    random_state=42
)
dt.fit(X_tr2, y_tr2)
y_pred_dt = dt.predict(X_te2)
y_prob_dt = dt.predict_proba(X_te2)[:, 1]

print('=== Árvore de Decisão — Relatório de Classificação ===')
print(classification_report(y_te2, y_pred_dt, target_names=['Sem Risco', 'Risco Incêndio']))
print(f'AUC-ROC: {roc_auc_score(y_te2, y_prob_dt):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Matriz de confusão
cm2 = confusion_matrix(y_te2, y_pred_dt)
ConfusionMatrixDisplay(cm2, display_labels=['Sem Risco', 'Risco']).plot(ax=axes[0], colorbar=False, cmap='Oranges')
axes[0].set_title('Matriz de Confusão — Árvore de Decisão')

# Importância das features
feat_imp = pd.Series(dt.feature_importances_, index=FEAT_DT).sort_values(ascending=True)
feat_imp.plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_xlabel('Importância (Gini)')
axes[1].set_title('Importância das Variáveis — Risco de Incêndio')

plt.tight_layout()
plt.savefig('../outputs/figures/03_arvore_resultados.png')
plt.show()

In [ ]:
# Visualização da Árvore
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt, feature_names=FEAT_DT, class_names=['Sem Risco', 'Risco'],
          filled=True, rounded=True, max_depth=3, ax=ax, fontsize=9)
ax.set_title('Árvore de Decisão — Risco de Incêndio (primeiros 3 níveis)', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/03_arvore_visualizacao.png', bbox_inches='tight')
plt.show()

In [ ]:
# Regras de decisão em linguagem natural
print('=== Regras de Decisão da Árvore (linguagem natural) ===\n')
rules_text = export_text(dt, feature_names=FEAT_DT, max_depth=4)
print(rules_text[:3000])

joblib.dump(knn_final, '../outputs/models/knn_onda_calor.pkl')
joblib.dump(scaler_knn, '../outputs/models/scaler_knn.pkl')
joblib.dump(dt, '../outputs/models/arvore_incendio.pkl')
print('\nModelos salvos em outputs/models/')